# 心電図のピークを見つける

心電図のデータから、心臓が打った瞬間のピーク（山のてっぺん。心電図では R）を見つける手順を、試しながら考えます。

- **Python を知らなくてもできます。** 変えるのは、`←` の付いた数字と `True` / `False` だけです
- セルを実行する: セルをクリックして `Shift + Enter`（上から順に実行します）
- 結果は、プリントに書き込みます
- データは**合成データ**です（実際の人の計測ではありません）

## 準備

下のセルを 1 回実行してください（中身は読まなくてかまいません）。
グラフが 1 つ出れば準備完了です。最初だけ、少し時間がかかります。

In [ ]:
import os

import matplotlib.pyplot as plt

DATA_URL = "https://misaki113.github.io/kamiyama-signal/data/"  # Colab などデータが手元にない環境では、配布ページの URL + "data/" を入れる
BEATS = 34  # このデータで、心臓が実際に打った回数


def data_path(name):
    """データファイルの場所を探す。見つからなければ DATA_URL から取ってくる。"""
    for folder in ["data", "../data", "."]:
        path = os.path.join(folder, name)
        if os.path.exists(path):
            return path
    if DATA_URL != "":
        import urllib.request

        urllib.request.urlretrieve(DATA_URL + name, name)
        return name
    raise FileNotFoundError(name + " が見つかりません。data フォルダの場所を確かめてください")


def load_ecg(name):
    """CSV を読んで、サンプリング周波数 fs（1 秒あたりの点の数）と、心電図の配列を返す。"""
    fs = 0.0
    ecg = []
    header_done = False
    with open(data_path(name)) as f:
        for line in f:
            line = line.strip()
            if line.startswith("#"):  # メタデータの行
                key, value = line[1:].split(":", 1)
                if key.strip() == "sampling_rate_hz":
                    fs = float(value)
            elif not header_done:  # 列名の行
                header_done = True
            else:  # データの行
                ecg.append(float(line.split(",")[1]))
    return fs, ecg


def moving_average(x, k):
    """各点を「その点を中心とした k 個の平均」に置きかえる。累積和を使うので O(n)。"""
    n = len(x)
    h = k // 2
    c = [0.0]
    for value in x:
        c.append(c[-1] + value)
    y = []
    for i in range(n):
        lo = max(0, i - h)
        hi = min(n - 1, i + h)
        y.append((c[hi + 1] - c[lo]) / (hi - lo + 1))
    return y


def find_peaks(x, fs, threshold, top_only, refractory_ms, remove_drift):
    """ピークを見つける。使う手順は、引数で切りかえる。"""
    n = len(x)

    # 手順 0: ゆっくりした揺れを求めて、引く
    if remove_drift:
        smooth = moving_average(x, 9)  # 細かいノイズをならす
        drift = moving_average(x, int(0.6 * fs) + 1)  # 0.6 秒の平均 = ゆっくりした揺れ
        x = [smooth[i] - drift[i] for i in range(n)]

    # 手順 1: しきい値を超えた点を、すべて拾う
    theta = threshold * max(x)
    peaks = []
    for i in range(n):
        if x[i] > theta:
            peaks.append(i)

    # 手順 2: ピーク（上りから下りに変わる点）だけを残す
    if top_only:
        tops = []
        for i in peaks:
            if 0 < i < n - 1 and x[i - 1] < x[i] and x[i] >= x[i + 1]:
                tops.append(i)
        peaks = tops

    # 手順 3: 直前のピークから refractory_ms ミリ秒以内のピークは、同じ拍とみなす（高い方を残す）
    if refractory_ms > 0:
        gap = refractory_ms / 1000 * fs
        kept = []
        for i in peaks:
            if len(kept) == 0 or i - kept[-1] > gap:
                kept.append(i)
            elif x[i] > x[kept[-1]]:
                kept[-1] = i
        peaks = kept

    return x, theta, peaks


def try_it(name, threshold=0.5, top_only=False, refractory_ms=0, remove_drift=False,
           t_from=2.0, t_to=10.0, show_rate=False):
    """データを読み、ピークを見つけて、グラフと数を出す。"""
    fs, ecg = load_ecg(name)
    x, theta, peaks = find_peaks(ecg, fs, threshold, top_only, refractory_ms, remove_drift)
    t = [i / fs for i in range(len(x))]

    plt.figure(figsize=(11, 3))
    plt.plot(t, x, linewidth=1)
    plt.axhline(theta, color="orange", linestyle="--")
    plt.plot([t[i] for i in peaks], [x[i] for i in peaks], "rv")
    plt.xlim(t_from, t_to)
    plt.xlabel("time [s]")
    plt.ylabel("[mV]")
    plt.title(name)
    plt.show()

    rates = [60 / ((peaks[k + 1] - peaks[k]) / fs) for k in range(len(peaks) - 1)]
    if show_rate and len(rates) > 0:
        plt.figure(figsize=(11, 2.2))
        plt.plot([t[i] for i in peaks[1:]], rates, "o-")
        plt.xlabel("time [s]")
        plt.ylabel("heart rate [bpm]")
        plt.show()

    print("見つかったピーク:", len(peaks), "個　（心臓が打ったのは", BEATS, "回）")
    if len(peaks) > 1:
        seconds = (peaks[-1] - peaks[0]) / fs  # 最初のピークから最後のピークまでの時間
        print("平均の心拍数:", round(60 * (len(peaks) - 1) / seconds), "回／分")


try_it("ecg_clean.csv", threshold=2.0)  # しきい値をうんと高くして、まずは波形だけを見る

## ワーク 1　しきい値を超えた点を拾う

オレンジの破線が**しきい値**、赤い ▼ が「ピーク」として拾われた点です。
`threshold = 0.5` は、「いちばん高いピークの 0.5 倍の高さに線を引く」という意味です。

1. まず、そのまま実行します。ピークはいくつ見つかりましたか
2. `0.5` をほかの数字（0.3、0.7、0.9 など）に変えて、実行し直します。34 個にできますか

In [ ]:
threshold = 0.5  # ← この数字を変える

try_it("ecg_clean.csv", threshold=threshold)

数が合わないときは、R を 1 つだけ拡大して見ます。何が起きていますか。

In [ ]:
try_it("ecg_clean.csv", threshold=threshold, t_from=2.0, t_to=2.2)

## ワーク 2　R 1 つにつき、ピークを 1 点だけ選ぶ

「上りから下りに変わる点だけを残す」という手順を足します。
`top_only` を `False` から `True` に書きかえて、実行してください。

In [ ]:
top_only = False  # ← True に変える

try_it("ecg_clean.csv", threshold=0.5, top_only=top_only)

## ワーク 3　ノイズのあるデータで試す

同じ手順を、ノイズが乗ったデータ `ecg_noisy.csv` に使います。

1. まず、そのまま実行します。ピークはいくつ見つかりましたか。どこで余分に拾っていますか
2. 「ピークを 1 つ見つけたら、そのあと `refractory_ms` ミリ秒は探さない」という手順を足します。
   `0` を `200` に変えて、実行します
3. `1000` にすると、どうなりますか

In [ ]:
refractory_ms = 0  # ← この数字を変える（0 は「この手順を使わない」）

try_it("ecg_noisy.csv", threshold=0.5, top_only=True, refractory_ms=refractory_ms)

余分に拾っている所を拡大して見るには、下のセルの `t_from` と `t_to`（秒）を変えます。

In [ ]:
try_it("ecg_noisy.csv", threshold=0.5, top_only=True, refractory_ms=refractory_ms, t_from=2.0, t_to=2.2)

## ワーク 4　手ごわいデータ

`ecg_hard.csv` は、全体がゆっくり上下に揺れています。

1. まず、そのまま実行します。ピークはいくつ見つかりましたか。見落としと、拾いすぎは、どこにありますか
2. どうすれば直せるかを、プリントに書いてください
3. 書けたら、`remove_drift` を `True` に変えて実行します。何が変わりましたか

In [ ]:
remove_drift = False  # ← 自分の案を書いてから、True に変える

try_it("ecg_hard.csv", threshold=0.5, top_only=True, refractory_ms=200, remove_drift=remove_drift,
       t_from=9.0, t_to=21.0, show_rate=True)

## 自由に試す

データ、数字、`True` / `False` を、好きに組み合わせて試せます。

- 手順を 1 つだけ外すと、どうなりますか
- `refractory_ms` を `600` にすると、ピークが 1 つ減ります。消えたのは、どのピークですか
- 17 秒あたりで、心拍数が 1 回だけ大きくはね上がります。これは手順の誤りでしょうか。波形に戻って確かめてください
- ピークを見つける手順の中身は、「準備」のセルの `find_peaks` に書いてあります。読める人は、書きかえてもかまいません

In [ ]:
try_it(
    "ecg_hard.csv",  # ecg_clean.csv / ecg_noisy.csv / ecg_hard.csv
    threshold=0.5,
    top_only=True,
    refractory_ms=200,
    remove_drift=True,
    t_from=14.0,
    t_to=20.0,
    show_rate=True,
)